# Edge situational awareness: GPU training on Kaggle

Trains the three models of [edge-situational-awareness](https://github.com/mustafarkhan/edge-situational-awareness)
properly on a free Kaggle GPU: **sea** (SeaDronesSee swimmers and boats), **thermal** (HIT-UAV people and vehicles
at night) and **land** (VisDrone people and vehicles). It downloads the datasets itself, uses the same data splits as
the repo, and writes `esa_gpu_results.zip` (weights + training logs) to the Output tab.

**Before running:** Settings (right panel) → Accelerator **GPU T4 x2** (or P100), **Internet on**.
Then **Save Version → Save & Run All (Commit)** so it runs in the background (up to 12 h), even with this tab closed.

Evaluation is not done here: download the zip and score it with the repository's evaluation scripts, so every number is measured with the same scripts
as the rest of the README.

In [ ]:
# Settings
STAGES = "sea thermal land sea1280"  # order = priority; later stages are skipped if time runs out
QUICK = 1  # 1 = one short epoch per model, to test the setup first (~30 min); then set 0 for the full run
SDS_SHARE = ""  # only if the sea download fails: the 'Object Detection v2' link from
# https://seadronessee.cs.uni-tuebingen.de/dataset

In [ ]:
import os
import shutil
import subprocess
import urllib.request

# 1. check the session settings before anything else, with a clear message
try:
    urllib.request.urlopen("https://github.com", timeout=10)
except Exception as e:
    raise SystemExit(
        f"No internet ({e}).\nRight panel -> Session options -> Internet: ON (needs a phone-verified account), then run again."
    )
if shutil.which("nvidia-smi") is None:
    raise SystemExit("No GPU. Right panel -> Session options -> Accelerator: GPU T4 x2, then run again.")
!nvidia-smi --query-gpu=name,memory.total --format=csv

# 2. get the code and run everything
!rm -rf /tmp/esa_repo && git clone -q --depth 1 https://github.com/mustafarkhan/edge-situational-awareness /tmp/esa_repo
assert os.path.isdir("/tmp/esa_repo/scripts"), "git clone failed: see the message above"
env = dict(os.environ, STAGES=STAGES, QUICK=str(QUICK), OUT="/kaggle/working")
if SDS_SHARE:
    env["SDS_SHARE"] = SDS_SHARE
p = subprocess.run(["bash", "scripts/kaggle/run_kaggle.sh"], cwd="/tmp/esa_repo", env=env)
print("exit code", p.returncode)

In [ ]:
# Summary: best validation mAP of each run (validation split only; test is scored later, outside Kaggle)
import glob

import pandas as pd

for f in sorted(glob.glob("/kaggle/working/esa_gpu/*/results.csv")):
    d = pd.read_csv(f)
    d.columns = [c.strip() for c in d.columns]
    best = d.loc[d["metrics/mAP50-95(B)"].idxmax()]
    print(
        f"{f.split('/')[-2]:16s} epochs {len(d):3d}  best val mAP50-95 {best['metrics/mAP50-95(B)']:.3f}"
        f"  mAP50 {best['metrics/mAP50(B)']:.3f}  (epoch {int(best['epoch'])})"
    )
!ls -la /kaggle/working/esa_gpu_results.zip